In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import argparse
import os
from typing import Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import periodogram
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.seasonal import seasonal_decompose


# -----------------------------
# Helpers de parsing e limpeza
# -----------------------------
def parse_intervalo_to_start_end(intervalo: str) -> Tuple[pd.Timedelta, pd.Timedelta]:
    """
    Converte "HH:MM:SS a HH:MM:SS" (aceita variações com espaços/acentos) em (start,end).
    """
    txt = intervalo.strip()
    txt = txt.replace("à", "a").replace("às", "a")  # tolera acentos
    partes = [p.strip() for p in txt.split("a")]
    if len(partes) != 2:
        raise ValueError(f"Intervalo inválido: {intervalo!r}")
    start = pd.to_timedelta(partes[0])
    end = pd.to_timedelta(partes[1])
    return start, end



def build_timestamp(row, date_col: str = "Data", intervalo_col: str = "Intervalo") -> pd.Timestamp:
    """
    Usa Data + início do intervalo como timestamp (ponto representativo).
    """
    base_date = row[date_col]
    start_td, _ = parse_intervalo_to_start_end(row[intervalo_col])
    return base_date + start_td

def load_and_prepare(
    csv_path: str,
    sep: str = ",",
    date_col: str = "Data",
    intervalo_col: str = "Intervalo",
    value_col: str = "Vazao",
    missing_sentinel: float = -1,
    dayfirst: bool = True,   # <-- novo
) -> pd.DataFrame:
    """
    Carrega CSV, parseia data e intervalo, cria timestamp, trata faltantes e ordena.
    """
    df = pd.read_csv(csv_path, sep=sep)

    # Normaliza colunas esperadas
    for c in [date_col, intervalo_col, value_col]:
        if c not in df.columns:
            raise KeyError(f"Coluna esperada não encontrada: {c!r}. Colunas: {list(df.columns)}")

    # --- Limpezas robustas de Data/Intervalo ---
    df[date_col] = df[date_col].astype(str).str.strip().str.replace("\u00A0", " ", regex=False)  # NBSP
    df[date_col] = df[date_col].str.replace("/", "-", regex=False)  # "28/4/2023" -> "28-4-2023"
    df[intervalo_col] = df[intervalo_col].astype(str).str.strip()

    # Converte Data para datetime (aceita "28-4-2023", "28-04-2023", etc.)
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=dayfirst, errors="coerce")

    if df[date_col].isna().any():
        bad = df.loc[df[date_col].isna(), date_col].astype(str).unique()[:5]
        raise ValueError(
            "Falha no parse da coluna de Data. Exemplos problemáticos: "
            + ", ".join(map(str, bad))
            + ". Tente ajustar separador (--sep), codificação, ou passe dayfirst corretamente."
        )

    # Cria timestamp a partir do início do intervalo
    df["timestamp"] = df.apply(lambda r: build_timestamp(r, date_col, intervalo_col), axis=1)

    # Converte valor e marca faltantes
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df[value_col] = df[value_col].where(df[value_col] != missing_sentinel, np.nan)

    # Ordena e remove duplicatas de timestamp (se houver, mantém primeiro)
    df = df.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="first").reset_index(drop=True)

    return df[["timestamp", value_col, date_col, intervalo_col]]



# -----------------------------
# Estimativas de frequência/periodicidade
# -----------------------------
def estimate_sampling_delta_seconds(ts: pd.Series) -> Optional[float]:
    """
    Estima delta modal (em segundos) entre amostras consecutivas.
    Retorna None se não houver pelo menos 2 timestamps.
    """
    if ts.size < 2:
        return None
    deltas = ts.diff().dropna().dt.total_seconds()
    if deltas.empty:
        return None
    # Moda robusta: arredonda a minutos para consolidar variações pequenas
    rounded = (deltas / 60.0).round().astype(int)
    if rounded.empty:
        return None
    mode_min = rounded.mode().iloc[0]
    return float(mode_min * 60.0)


def estimate_period_acf(values: pd.Series, max_lag: Optional[int] = None) -> Optional[int]:
    """
    Estima período (em #amostras) pelo primeiro pico local do ACF (excluindo lag=0).
    """
    x = values.values.astype(float)
    # precisa de amostras suficientes
    n = np.isfinite(x).sum()
    if n < 8:
        return None

    # preenche faltantes de forma simples (forward-fill) para ACF (opcional)
    x_pd = values.copy()
    x_pd = x_pd.ffill().bfill()

    # define max_lag
    if max_lag is None:
        max_lag = min(7 * 24, max(10, len(x_pd) // 2))  # limite razoável

    acf_vals = acf(x_pd, nlags=max_lag, fft=True)
    # procura primeiro pico após lag=0
    # critério simples: valor localmente maior que vizinhos
    peaks = []
    for k in range(2, len(acf_vals) - 1):
        if acf_vals[k] > acf_vals[k - 1] and acf_vals[k] > acf_vals[k + 1]:
            peaks.append((k, acf_vals[k]))
    if not peaks:
        return None
    # escolhe o pico com maior ACF (entre os primeiros)
    peaks.sort(key=lambda t: t[1], reverse=True)
    return int(peaks[0][k := 0])  # número de amostras


def estimate_period_fft(values: pd.Series) -> Optional[int]:
    """
    Estima período (em #amostras) via pico do periodograma (excluindo frequência zero).
    """
    x = values.copy().astype(float)
    # preenchimento simples para lidar com NaN no FFT
    x = x.ffill().bfill()
    if len(x) < 8:
        return None

    fs = 1.0  # 1 amostra por passo (índice)
    freqs, power = periodogram(x, fs=fs, scaling="spectrum", detrend="linear")
    # ignora freq=0
    mask = freqs > 0
    if not mask.any():
        return None
    freqs_nz = freqs[mask]
    power_nz = power[mask]
    idx = np.argmax(power_nz)
    f_peak = freqs_nz[idx]
    if f_peak <= 0:
        return None
    period_samples = int(round(1.0 / f_peak))
    return period_samples if period_samples >= 2 else None


# -----------------------------
# Plots
# -----------------------------
def ensure_figdir(path: str = "./figs6"):
    os.makedirs(path, exist_ok=True)
    return path


def plot_time_series(df: pd.DataFrame, value_col: str, figdir: str):
    fig, ax = plt.subplots(figsize=(12, 4))
    # linha base (com NaN vai quebrar)
    ax.plot(df["timestamp"], df[value_col], lw=1.5)
    # marca NaN
    nan_mask = df[value_col].isna()
    ax.scatter(df.loc[nan_mask, "timestamp"], [np.nan] * nan_mask.sum(), s=0)  # placeholder
    ax.set_title("Série temporal (com faltantes)")
    ax.set_xlabel("Tempo")
    ax.set_ylabel(value_col)
    ax.grid(True, alpha=0.3)
    out = os.path.join(figdir, "01_serie_temporal.png")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def plot_missing_map(df: pd.DataFrame, value_col: str, figdir: str):
    """
    Mapa simples de faltantes ao longo do tempo (1 = faltante, 0 = presente).
    """
    fig, ax = plt.subplots(figsize=(12, 2.5))
    miss = df[value_col].isna().astype(int).values.reshape(1, -1)
    ax.imshow(miss, aspect="auto", interpolation="nearest")
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_title("Mapa de dados faltantes (1=faltante)")
    out = os.path.join(figdir, "02_missing_map.png")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def plot_acf(values: pd.Series, figdir: str, max_lag: Optional[int] = None):
    # preenchimento simples
    x = values.ffill().bfill()
    n = len(x)
    if max_lag is None:
        max_lag = min(7 * 24, max(10, n // 2))
    acf_vals = acf(x, nlags=max_lag, fft=True)

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.stem(range(len(acf_vals)), acf_vals, basefmt=" ")
    ax.set_title("Autocorrelação (ACF)")
    ax.set_xlabel("Lag (nº de amostras)")
    ax.set_ylabel("ACF")
    ax.grid(True, alpha=0.3)
    out = os.path.join(figdir, "03_acf.png")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def plot_decompose(df: pd.DataFrame, value_col: str, period: int, figdir: str):
    # para decomposição, preencher temporariamente
    x = df[value_col].ffill().bfill()
    try:
        res = seasonal_decompose(x, period=period, model="additive", two_sided=True, extrapolate_trend="freq")
    except Exception as e:
        print(f"[WARN] Falha na decomposição sazonal (period={period}): {e}")
        return

    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(df["timestamp"], x)
    axes[0].set_title("Decomposição sazonal - Observado")
    axes[1].plot(df["timestamp"], res.trend)
    axes[1].set_title("Trend")
    axes[2].plot(df["timestamp"], res.seasonal)
    axes[2].set_title("Seasonal")
    axes[3].plot(df["timestamp"], res.resid)
    axes[3].set_title("Residual")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Tempo")
    out = os.path.join(figdir, f"04_decompose_period_{period}.png")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def plot_periodogram(values: pd.Series, figdir: str):
    x = values.ffill().bfill()
    fs = 1.0
    freqs, power = periodogram(x, fs=fs, scaling="spectrum", detrend="linear")
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(freqs[1:], power[1:])  # ignora freq zero
    ax.set_title("Periodograma (FFT) — pico indica frequência dominante")
    ax.set_xlabel("Frequência (ciclos por amostra)")
    ax.set_ylabel("Potência")
    ax.grid(True, alpha=0.3)
    out = os.path.join(figdir, "05_periodogram.png")
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


# -----------------------------
# Main (CLI)
# -----------------------------
def main():


    df = load_and_prepare(
    csv_path='jisa_experiments/resultado2.csv',
    sep=',',
    date_col='Data',
    intervalo_col='Intervalo',
    value_col='Vazao',
    missing_sentinel=-1.0,
    dayfirst=True,  # <-- importante p/ "28-4-2023"
)


    figdir = ensure_figdir()

    # Plots básicos
    plot_time_series(df, 'Vazao', figdir)
    plot_missing_map(df, 'Vazao', figdir)
    plot_acf(df['Vazao'], figdir)
    plot_periodogram(df['Vazao'], figdir)

    # Estimativas
    delta_sec = estimate_sampling_delta_seconds(df["timestamp"])
    period_acf = estimate_period_acf(df['Vazao'])
    period_fft = estimate_period_fft(df['Vazao'])

    print("\n===== Estimativas =====")
    if delta_sec is not None:
        print(f"Delta modal entre amostras: {delta_sec/3600:.2f} horas ({delta_sec:.0f} s)")
    else:
        print("Delta modal: indisponível (amostras insuficientes).")

    if period_acf is not None:
        print(f"Período (ACF): {period_acf} amostras")
    else:
        print("Período (ACF): não estimado.")

    if period_fft is not None:
        print(f"Período (FFT): {period_fft} amostras")
    else:
        print("Período (FFT): não estimado.")

    # Decide um período para decomposição: prioriza ACF, senão FFT
    period_for_decomp = None
    for candidate in (period_acf, period_fft):
        if candidate is not None and candidate >= 2 and candidate <= max(2, len(df) // 2):
            period_for_decomp = candidate
            break

    if period_for_decomp is not None:
        plot_decompose(df, 'Vazao', period_for_decomp, figdir)
        print(f"Decomposição sazonal gerada com period={period_for_decomp}.")
    else:
        print("Decomposição sazonal: não realizada (período inválido ou não estimado).")

    print(f"\nFiguras salvas em: {os.path.abspath(figdir)}")


if __name__ == "__main__":
    main()


ValueError: Falha no parse da coluna de Data. Exemplos problemáticos: NaT. Tente ajustar separador (--sep), codificação, ou passe dayfirst corretamente.

In [11]:
!pip install matplotlib pandas numpy scipy statsmodels